## Setup

This notebook walks through `pydantic-ai`'s core primitives: schemas, agents, structured output, dependency injection, the five output modes, run methods, and a function tool. Cells build on each other; run them in order.

Output uses `rprint` from `rich`, which formats Python objects (models, dicts, exceptions) more readably than the built-in `print`.

In [ ]:
from rich.markdown import Markdown
from rich import print as rprint

rprint(Markdown("# Introduction to pydantic-ai"))
rprint({"user": "Alice", "question": "What is this?"})

## Pydantic models

Three schemas the rest of the notebook reuses:

- **`Question`** — the input shape, with length constraints.
- **`Citation`** — a pointer to a sentence or paragraph in a source document.
- **`Answer`** — the output shape: text, optional citations, a confidence score, and an escalation flag.

`Answer` declares two validators:

- A field-level validator that rejects empty citation quotes.
- A model-level validator that rejects low-confidence answers whose text doesn't hedge.

The same `ValidationError` fires whether the input came from your own code or from an LLM's tool call.

TODO: Mention inbuild validators like EmailStr and maybe strictnesss

In [ ]:
from __future__ import annotations

from typing import Annotated, Literal

from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator


class Question(BaseModel):
    text: Annotated[str, Field(min_length=3, max_length=2000)]
    user_id: str | None = Field(
        default=None,
        description="Optional user identifier for the session.",
    )


class Citation(BaseModel):
    """A pointer to a sentence or paragraph in a source document."""

    doc_id: Annotated[str, Field(min_length=1, max_length=100)]
    quote: Annotated[str, Field(min_length=1, max_length=500)]


class Answer(BaseModel):
    """A structured assistant response.

    `risk_flag` lets the assistant signal escalation needs (out-of-scope
    or sensitive queries, urgent issues) without burying that signal in
    the response text.
    """

    text: Annotated[str, Field(min_length=1, max_length=4000)]
    citations: list[Citation] = Field(default_factory=list)
    confidence: Annotated[float, Field(ge=0.0, le=1.0)]
    risk_flag: Literal["none", "escalate", "urgent"] = "none"

    @field_validator("citations")
    @classmethod
    def reject_empty_citation_quotes(cls, v: list[Citation]) -> list[Citation]:
        if any(not c.quote.strip() for c in v):
            raise ValueError("citation quotes must not be empty or whitespace")
        return v

    @model_validator(mode="after")
    def low_confidence_must_admit_uncertainty(self) -> "Answer":
        text_l = self.text.lower()
        if self.confidence < 0.4 and "not sure" not in text_l and "may" not in text_l:
            raise ValueError(
                "confidence < 0.4 — answer text must signal uncertainty "
                '(e.g. contain "not sure" or "may")'
            )
        return self

## Validators run at construction time

Construct `Answer` directly:

- The first call satisfies both validators.
- The second deliberately trips `low_confidence_must_admit_uncertainty` (confidence `0.1` paired with confident-sounding text).

The `ValidationError` printed in the second case is exactly what an agent receives when its draft is rejected — and feeds back to the model on retry.

In [ ]:
# Happy path
ok = Answer(text="Standard shipping arrives in 3-5 business days.", confidence=0.95)
rprint("valid:", ok)

# Sad path — confidence is low but the text doesn't hedge
try:
    Answer(text="It costs 5 dollars.", confidence=0.1)
except ValidationError as exc:
    rprint("\nValidationError raised, as expected:")
    rprint(exc)

## Resolve the project root

The `Settings` class (next cell) loads from `.env`. Compute the project root from the notebook's working directory so the lookup works whether the kernel was started in `notebooks/` or the repo root.

In [ ]:
from pathlib import Path

PROJECT_ROOT_PATH = Path.cwd().parent.parent
rprint(PROJECT_ROOT_PATH)

c:\Users\TomášZítka\DataSenticsProjects\CSOB\pydantic_ai_workshop

## Typed configuration with `pydantic-settings`

`BaseSettings` reads from environment variables and `.env`, validates types, and raises on missing required fields. One `Settings()` call gives you autocompleted access to every config knob.

Benefits over scattered `os.getenv(...)` calls:

- Types are enforced — `int`, `Literal`, `Path`, etc.
- Defaults live next to the field declarations.
- Missing required values raise *at construction*, not when the first use happens deep in a request.

In [ ]:
from pathlib import Path

from pydantic_settings import BaseSettings, SettingsConfigDict

_ENV_FILE = PROJECT_ROOT_PATH / ".env"


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=_ENV_FILE,
        env_file_encoding="utf-8",
        extra="ignore",
    )

    anthropic_api_key: str = Field(
        ...,
        description="Workshop-allocated Anthropic API key. Required.",
    )
    model_id: str = Field(
        default="claude-haiku-4-5-20251001",
        description="Default model id. Haiku is cheap and fast for the demos.",
    )
    log_level: Literal["DEBUG", "INFO", "WARNING", "ERROR"] = "INFO"


settings = Settings()  # type: ignore[call-arg]

rprint("model_id  :", settings.model_id)
rprint("log_level :", settings.log_level)
rprint("api key   :", bool(settings.anthropic_api_key))

## Build the model

`AnthropicModel` wraps the provider (which holds credentials) and the model id. Every `Agent(...)` constructor in the rest of the notebook receives this `model` instance.

In [ ]:
from pydantic_ai.models.anthropic import AnthropicModel
from pydantic_ai.providers.anthropic import AnthropicProvider

model = AnthropicModel(
    settings.model_id,
    provider=AnthropicProvider(api_key=settings.anthropic_api_key),
)

## Optional — log HTTP traffic

Uncomment to log every HTTP request the Anthropic SDK sends. The output is verbose, but it's the cleanest way to see a multi-step tool-calling loop on the wire.

In [ ]:
# import logging
# logging.basicConfig()
# logging.getLogger("anthropic").setLevel(logging.DEBUG)
# logging.getLogger("httpx").setLevel(logging.DEBUG)

## Smallest possible agent

A model plus an instruction string. With no `output_type` argument the default applies (`str`), so `result.output` is the plain text the model returned.

In [ ]:
from pydantic_ai import Agent

trivial = Agent(model, instructions="You are a helpful tutor. Be brief.")

result = await trivial.run("What is 2 + 2?")
rprint("type(result.output):", type(result.output).__name__)
rprint("result.output:", result.output)

## Structured output with `output_type=Answer`

Setting `output_type=Answer` makes `pydantic-ai` validate the LLM's reply against the schema before returning. If validation fails the agent retries, feeding the validation error back to the model so it can correct the output.

`result.output` is then a typed `Answer` instance: attribute access (`.confidence`, `.risk_flag`, `.citations`) works directly without `json.loads` or `dict` indexing.

In [ ]:
from pydantic_ai import RunContext

assistant = Agent[str, Answer](
    model,
    output_type=Answer,
    deps_type=str,
    instructions=(
        "You are a helpful customer support assistant. "
        "Always answer in the structured Answer format. "
        "If you are not sure, set confidence below 0.4 and say so. "
        "If the question is sensitive or out of scope, "
        "set risk_flag='escalate'. "
        "If the question concerns an urgent issue, "
        "set risk_flag='urgent'."
    ),
)


@assistant.instructions
def greet_by_name(ctx: RunContext[str]) -> str:
    return (
        f"The user in this session is **{ctx.deps}**. "
        "Address them by first name once at the start of your reply."
    )

## One call

Invoke the agent. By the time this line returns, `result.output` is an `Answer` whose validators already passed; if the model had failed twice in a row the call would have raised.

In [ ]:
result = await assistant.run("What is your return policy?", deps="Alice")

rprint("isinstance(result.output, Answer):", isinstance(result.output, Answer))
rprint("text:", result.output.text)
rprint("confidence:", result.output.confidence)
rprint("risk_flag:", result.output.risk_flag)
rprint("citations:", result.output.citations)

## Dependency injection via `deps_type`

The agent declared `deps_type=str`, so every call must pass `deps=`. Inside the agent, `RunContext` exposes the value as `ctx.deps`.

The `@assistant.instructions` decorator below the agent declaration registers a *per-run* callback. It receives the `RunContext` for the current call and returns a string fragment that joins onto the static instructions. Per-run state lives in `deps`, not in a prompt rebuilt by hand every time.

In [ ]:
for user in ("Alice", "Bob"):
    result = await assistant.run("Can you remind me how shipping works?", deps=user)
    rprint(f"--- deps={user!r} ---")
    rprint(result.output.text)
    rprint()

## Output modes side-by-side

Five `output_type` shapes against the same `CityLocation` schema and the same prompt:

| Mode | Notes |
|---|---|
| `str` | Plain text. No schema. |
| `ToolOutput` | Default. Schema-validated via a synthetic output tool. |
| `NativeOutput` | Provider-side JSON mode (one fewer round-trip on Anthropic). | NOTE: "on Anthropic"?
| `PromptedOutput` | Schema described in the prompt — fallback for weaker models. |
| `TextOutput(parser)` | Free-form text parsed by your own function. |

The next cell runs all five and prints `type(result.output)` for each.

In [ ]:
from pydantic_ai import NativeOutput, PromptedOutput, TextOutput, ToolOutput


class CityLocation(BaseModel):
    """Tiny structured shape used by every demo agent below."""

    city: str
    country: str


PROMPT = "Where were the 2012 Olympics held?"


def parse_csv(text: str) -> CityLocation:
    """Parse 'city, country' text into a CityLocation."""
    parts = [p.strip() for p in text.split(",", 1)]
    if len(parts) != 2:
        raise ValueError(f"expected 'city, country', got {text!r}")
    return CityLocation(city=parts[0], country=parts[1])


_instructions = (
    "Answer the user's geography question. When asked for a CSV string, "
    "respond with exactly 'city, country' and nothing else."
)

mode_agents: dict[str, Agent] = {
    "str (raw text)": Agent(model, output_type=str, instructions=_instructions),
    "ToolOutput (default)": Agent(
        model, output_type=ToolOutput(CityLocation), instructions=_instructions
    ),
    "NativeOutput (provider-native JSON)": Agent(
        model, output_type=NativeOutput(CityLocation), instructions=_instructions
    ),
    "PromptedOutput (no-protocol fallback)": Agent(
        model, output_type=PromptedOutput(CityLocation), instructions=_instructions
    ),
    "TextOutput (custom parser)": Agent(
        model, output_type=TextOutput(parse_csv), instructions=_instructions
    ),
}

rprint(mode_agents)

## Run each mode

Same call shape for every entry: `await agt.run(PROMPT)`. The output type and value differ depending on which `output_type=` the agent was built with.

In [ ]:
rprint(f"Prompt: {PROMPT}\n")
for label, agt in mode_agents.items():
    result = await agt.run(PROMPT)
    rprint(f"--- {label} ---")
    rprint(f"type:  {type(result.output).__name__}")
    rprint(f"value: {result.output!r}\n")

## Run methods

Three ways to invoke the same agent:

- `agent.run_sync(...)` — blocks the calling thread. Demos and notebooks only.
- `await agent.run(...)` — async coroutine; the default in async code.
- `async with agent.run_stream(...) as result` — yields text deltas as the model emits them.

(`agent.iter(...)` for node-by-node inspection exists too; covered in a later notebook.)

In [ ]:
# run_sync — blocking
import nest_asyncio
nest_asyncio.apply()

sync_result = assistant.run_sync("Summarize the return policy in one sentence.", deps="Alice")
rprint("run_sync   ->", sync_result.output.text[:120], "...")


Note: Mention that all these cells with async code work thanks to the notebook, otherwise you need to setup async loop

In [ ]:
# await run — async, the default in async code
async_result = await assistant.run("Summarize the return policy in one sentence.", deps="Alice")
rprint("await run  ->", async_result.output.text[:120], "...")

In [ ]:
# run_stream — token-by-token. Use a str-output agent so the deltas print cleanly.
text_agent = Agent(
    model,
    output_type=str,
    instructions="You are a helpful support assistant. Give a friendly two-sentence answer.",
)

print("run_stream ->", end=" ")
async with text_agent.run_stream("How does standard shipping work?") as result:
    async for chunk in result.stream_text(delta=True):
        print(chunk, end="", flush=True)
print()

## Function tools

A plain Python function. Registering it on an agent (next cell) turns the function's signature and docstring into the tool schema the LLM sees. The model decides when to call the tool; `pydantic-ai` parses the call, runs the function, and feeds the result back into the conversation.

In [ ]:
_PRICE_LIST: dict[str, float] = {
    "basic_plan": 9.99,
    "pro_plan": 29.99,
    "enterprise_plan": 99.99,
    "addon_storage": 4.99,
    "addon_priority_support": 19.99,
}


def lookup_price(item_name: str) -> dict:
    """Look up a product price by canonical name. Returns USD amount or an error."""
    key = item_name.strip().lower()
    if key not in _PRICE_LIST:
        return {"error": "unknown item", "known": sorted(_PRICE_LIST)}
    return {"item_name": key, "price_usd": _PRICE_LIST[key]}

NOTE: Mention to "not consuming it" feels confusing, why emphasise it?

## `event_stream_handler=` — observe a run without consuming it

Pass an async callback as `event_stream_handler=` and the agent will fire it for every internal event — tool calls, tool results, model deltas — alongside the normal `await agent.run(...)` return value.

The handler below prints each tool call and its result as the loop runs.

In [ ]:
from collections.abc import AsyncIterable

from pydantic_ai.messages import (
    AgentStreamEvent,
    FunctionToolCallEvent,
    FunctionToolResultEvent,
)


async def print_tool_events(
    ctx: RunContext[None],
    events: AsyncIterable[AgentStreamEvent],
) -> None:
    async for event in events:
        if isinstance(event, FunctionToolCallEvent):
            print(f"[tool: {event.part.tool_name}({event.part.args})]")
        elif isinstance(event, FunctionToolResultEvent):
            print(f"[result: {event.part.content!r}]")


tool_agent: Agent[None, Answer] = Agent(
    model,
    output_type=Answer,
    instructions=(
        "You are a helpful assistant. Use the lookup_price tool when asked "
        "about a specific product price. Always answer in Answer format and "
        "cite the tool's return as a citation with doc_id='pricing'."
    ),
    tools=[lookup_price],
)

result = await tool_agent.run(
    "What does the pro_plan cost?",
    event_stream_handler=print_tool_events,
)
print(f"\nfinal: {result.output.text}")
print(f"conf : {result.output.confidence}")

## Recap

- Pydantic models declare the input/output shapes. Validators run on construction and during agent output validation.
- `pydantic-settings` replaces scattered `os.getenv(...)` calls with a typed `Settings` object.
- `Agent` ties together a model, instructions, an output schema, and (optionally) typed dependencies.
- `output_type=` chooses how the model's reply is decoded. `ToolOutput` is the default; the other four cover provider quirks and custom parsers.
- `deps_type=` + `@agent.instructions` keep per-run state out of the static prompt.
- Run methods: `run` for async code, `run_stream` for token-by-token UIs, `run_sync` for demos.
- `event_stream_handler=` exposes the agent's internal events without consuming the result.